# BBA Riemannian DSM — Phase 3 Re-run (post transpose-bug fix)

This notebook regenerates the BBA precomputed training data and trains `TangentScoreModel`
from scratch using the **fixed** `metric_tensor` (transpose permutation `(0,1,2,4,3,5)`).

**Why a fresh run?** The Phase 3.6 precomputed data was generated with the old buggy
`metric_tensor` (transpose `(0,1,2,5,3,4)`), which caused `s_log` to have ~1.3% vertical
leakage. That data must be discarded. See `THEORY.md §12` for the full root-cause analysis.

---

## Checklist before running

1. **GPU runtime**: Runtime → Change runtime type → T4 / A100
2. **Raw data on Drive**: `MyDrive/thesis-data/processed/bba_train.npy`,
   `bba_test.npy`, `bba_ref.npy` (uploaded from local machine — see §0b)
3. **Repo on Colab**: cloned in §1 (or set `REPO_ROOT` if you clone manually)

## Workflow

| Cell | What it does | Runtime |
|------|-------------|--------|
| §0 | Mount Drive, set paths | <1 s |
| §1 | Clone repo + install deps | ~2 min |
| §2 | Verify GPU + JAX | <1 s |
| §3 | Load data, verify no NaN | <1 s |
| §4 | **Regenerate precomputed data** (10 repeats) | ~85 min |
| §5 | Upload new precomputed data to Drive | ~10 min |
| §6 | Benchmark Phase B step | <1 min |
| §7 | **Train** (3000 epochs) | ~11 min |
| §8 | Loss curve | <1 s |
| §9 | Score sanity check (cosine similarity) | <1 min |
| §10 | Save checkpoint + results to Drive | <1 min |
| §11 | **Download instructions** (Drive → local) | (manual) |

## §0a. Mount Drive and set paths

In [18]:
from google.colab import drive
drive.mount("/content/drive")

import os

# ── Edit these if your Drive layout differs ────────────────────────────────
REPO_ROOT       = "/content"   # where the repo will be cloned
DRIVE_DATA      = "/content/drive/MyDrive/thesis-data"
PROCESSED_DIR   = f"{DRIVE_DATA}/processed"          # bba_train.npy, bba_test.npy, bba_ref.npy
PRECOMPUTED_DIR = f"{DRIVE_DATA}/precomputed/bba_v2" # noised_r00..r09.npz
DRIVE_CKPT      = f"{DRIVE_DATA}/runs/bba_phase3_v2" # checkpoint output
LOCAL_CKPT      = "/content/runs/bba_phase3_v2"      # Colab-local (fast I/O during training)
# ────────────────────────────────────────────────────────────────────────────

# ── Set to True only if you need to regenerate precomputed data from scratch ─
# Default: False — precomputed files are read directly from Drive (uploaded
# via rclone from local machine; see §0b for upload instructions).
REPROCESS_DATA = False
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(PRECOMPUTED_DIR, exist_ok=True)
os.makedirs(DRIVE_CKPT,      exist_ok=True)
os.makedirs(LOCAL_CKPT,      exist_ok=True)

print("Drive mounted.")
print(f"Processed data : {PROCESSED_DIR}")
print(f"Precomputed dir: {PRECOMPUTED_DIR}")
print(f"Checkpoint dir : {DRIVE_CKPT}")
print(f"REPROCESS_DATA : {REPROCESS_DATA}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.
Processed data : /content/drive/MyDrive/thesis-data/processed
Precomputed dir: /content/drive/MyDrive/thesis-data/precomputed/bba_v2
Checkpoint dir : /content/drive/MyDrive/thesis-data/runs/bba_phase3_v2
REPROCESS_DATA : False


## §0b. Upload data to Drive (local → Drive, one-time setup)

Run these **once from your local machine** before opening the notebook on Colab.
Both the raw processed arrays and the precomputed noised targets need to be on Drive.

### 1. Install and configure rclone (one-time)

```bash
brew install rclone          # macOS
rclone config                # follow prompts: New remote → Google Drive → OAuth
# name it "gdrive" when asked
```

### 2. Upload processed BBA arrays

```bash
rclone copy \
  /Users/I745505/private/thesis/pocs/manifold-scoremd/riemannian-scoremd/data/processed/ \
  gdrive:thesis-data/processed/ \
  --include "bba_*.npy" \
  --progress
```

Expected on Drive after upload:
- `thesis-data/processed/bba_train.npy`  — `(63000, 28, 3)` float32
- `thesis-data/processed/bba_test.npy`   — `(7000, 28, 3)` float32
- `thesis-data/processed/bba_ref.npy`    — `(28, 3)` float32

### 3. Upload precomputed noised targets (generated locally, fixed metric tensor)

```bash
rclone copy \
  /Users/I745505/private/thesis/pocs/manifold-scoremd/riemannian-scoremd/data/precomputed/bba/ \
  gdrive:thesis-data/precomputed/bba_v2/ \
  --progress
```

Expected on Drive: `thesis-data/precomputed/bba_v2/noised_r00.npz` … `noised_r09.npz`  
Each file is ~162 MB; total ~1.6 GB. rclone will resume interrupted uploads.

### Verify the upload

```bash
rclone ls gdrive:thesis-data/precomputed/bba_v2/
# Should list 10 files, each ~162 MB
```

Once these files are on Drive, run the notebook with `REPROCESS_DATA = False` (default) — §4 will be skipped and training starts immediately from §6.

## §1. Clone repo and install dependencies

In [19]:
import subprocess, sys

REPO_URL    = "https://github.com/grapentt/riemannian-scoremd.git"
REPO_BRANCH = "main"

REPO_DIR = "/content/manifold-scoremd"
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )
    print(f"Cloned → {REPO_DIR}")
else:
    print(f"Repo already present at {REPO_DIR}")

# Install Python deps not bundled with Colab
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "flax==0.10.4", "optax", "orbax-checkpoint>=0.11.20"],
    check=True
)

SRC_DIR = os.path.join(REPO_DIR, "src")
print(f"SRC_DIR: {SRC_DIR}  (exists: {os.path.isdir(SRC_DIR)})")

Repo already present at /content/manifold-scoremd
SRC_DIR: /content/manifold-scoremd/src  (exists: True)


## §2. Verify GPU, JAX, and imports

In [20]:
import sys, os, subprocess
import jax

# The repo at github.com/grapentt/riemannian-scoremd clones directly to
# /content/manifold-scoremd — src is at its root, not in a subdirectory.
_src_dir = "/content/manifold-scoremd/src"
if not os.path.isdir(_src_dir):
    raise RuntimeError(
        f"SRC_DIR not found: {_src_dir}\n"
        "Run §1 first to clone the repo."
    )
if _src_dir not in sys.path:
    sys.path.insert(0, _src_dir)
print(f"SRC_DIR: {_src_dir}  ✓")

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True
).stdout.strip()
print(f"GPU     : {gpu or 'NONE — switch to GPU runtime!'}")
print(f"JAX     : {jax.__version__}")
print(f"Devices : {jax.devices()}")

from manifold.pointcloud_jax import ShapeManifold
from diffusion.manifold_sde   import ManifoldVP
from models.tangent_mlp       import TangentScoreModel
from training.score_loss      import prepare_batch
from training.train_manifold  import train_from_precomputed, make_train_step
print("Imports OK")

SRC_DIR: /content/manifold-scoremd/src  ✓
GPU     : NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB
JAX     : 0.7.2
Devices : [CudaDevice(id=0)]
Imports OK


## §3. Load and verify data

In [21]:
import numpy as np
import jax.numpy as jnp

train_data = np.load(os.path.join(PROCESSED_DIR, "bba_train.npy"))  # (63000, 28, 3)
test_data  = np.load(os.path.join(PROCESSED_DIR, "bba_test.npy"))   # (7000, 28, 3)
x_ref      = np.load(os.path.join(PROCESSED_DIR, "bba_ref.npy"))    # (28, 3)

n, d = x_ref.shape
print(f"Train : {train_data.shape}   Test : {test_data.shape}")
print(f"n={n} Cα, d={d}, nd={n*d}")

assert not np.isnan(train_data).any(), "NaN in train_data!"
assert not np.isnan(test_data).any(),  "NaN in test_data!"
print("No NaN. Data OK.")

# Build manifold and SDE
manifold = ShapeManifold(dim=d, numpoints=n, alpha=1.0, base=x_ref)
sde      = ManifoldVP(manifold)

# Quick sanity check: metric_tensor kernel test on 3 frames
import jax
x_check = jnp.array(train_data[:3, None, :, :])  # (3, 1, n, d)
H = manifold.metric_tensor(x_check, asmatrix=True)  # (3, 1, nd, nd)

# Build a rotation generator and check H·v ≈ 0
x0 = train_data[0]  # (n, d)
v_rot = np.zeros((n, d), dtype=np.float32)
v_rot[:, 0] =  x0[:, 1]  # generator of rotation in x-y plane
v_rot[:, 1] = -x0[:, 0]
v_rot_flat = v_rot.reshape(-1)
H0 = np.array(H[0, 0])  # (nd, nd)
residual = np.linalg.norm(H0 @ v_rot_flat) / (np.linalg.norm(v_rot_flat) + 1e-12)

print(f"\nmetric_tensor kernel check (rotation generator):")
print(f"  ||H·v_rot|| / ||v_rot|| = {residual:.2e}  (expect < 1e-6 after bug fix)")
if residual > 1e-4:
    print("  !! LARGE RESIDUAL — check that the transpose fix is on the current branch !!")
else:
    print("  ✓ Fix confirmed.")

Train : (63000, 28, 3)   Test : (7000, 28, 3)
n=28 Cα, d=3, nd=84
No NaN. Data OK.

metric_tensor kernel check (rotation generator):
  ||H·v_rot|| / ||v_rot|| = 3.55e-06  (expect < 1e-6 after bug fix)
  ✓ Fix confirmed.


## §4. Regenerate precomputed training data (only if `REPROCESS_DATA = True`)

**Default**: skipped. Set `REPROCESS_DATA = True` in §0a to regenerate.

If `REPROCESS_DATA = False` the notebook reads files already on Drive
(`PRECOMPUTED_DIR`). Upload them from your local machine first — see §0b.

If `REPROCESS_DATA = True`:
- Runs `s_log` (Riemannian gradient, fixed metric tensor) for all 63k BBA frames × 10 repeats
- Writes directly to Drive so progress survives session restarts
- Skips repeats that already exist
- **Estimated time**: ~8–10 min/repeat × 10 ≈ 85 min on a T4

In [23]:
import time, glob

if not REPROCESS_DATA:
    precomp_files = sorted(glob.glob(os.path.join(PRECOMPUTED_DIR, "noised_r*.npz")))
    print(f"REPROCESS_DATA=False — skipping precomputation.")
    print(f"Found {len(precomp_files)} existing file(s) in {PRECOMPUTED_DIR}")
    if len(precomp_files) == 0:
        print("\n!! No precomputed files found. Either:")
        print("   1. Upload them from local machine (see §0b), or")
        print("   2. Set REPROCESS_DATA = True in §0a to regenerate.")
    else:
        s = np.load(precomp_files[0])
        print(f"  x_t:    {s['x_t'].shape}  ({s['x_t'].dtype})")
        print(f"  s_true: {s['s_true'].shape}")
        print(f"  t:      range [{s['t'].min():.2f}, {s['t'].max():.2f}]")
else:
    print("REPROCESS_DATA=True — regenerating precomputed data with s_log (fixed metric tensor).\n")

    # Warm up s_exp JIT
    print("Warming up s_exp JIT cache...")
    rng = jax.random.PRNGKey(0)
    rng, tk, nk = jax.random.split(rng, 3)
    x0_warm = jnp.array(train_data[:4])
    t_warm  = jax.random.uniform(tk, (4,), minval=T_MIN, maxval=T_MAX)
    t0 = time.time()
    prepare_batch(manifold, sde, x0_warm, t_warm, nk)
    print(f"  JIT warm-up: {time.time()-t0:.1f}s")

    # Time one batch to estimate total
    BATCH_PRECOMP = 128
    rng, tk, nk = jax.random.split(rng, 3)
    x0_b = jnp.array(train_data[:BATCH_PRECOMP])
    t_b  = jax.random.uniform(tk, (BATCH_PRECOMP,), minval=T_MIN, maxval=T_MAX)
    t0 = time.time()
    prepare_batch(manifold, sde, x0_b, t_b, nk)
    ms_per_sample = (time.time()-t0) / BATCH_PRECOMP * 1000
    est_total_min = ms_per_sample * len(train_data) * N_REPEATS / 1000 / 60
    print(f"  {ms_per_sample:.2f} ms/sample → estimated total: {est_total_min:.0f} min\n")

    def precompute_one_repeat(manifold, sde, train_data, t_min, t_max, rng, out_path, B=128):
        N, n, d = train_data.shape
        rng, t_key = jax.random.split(rng)
        t_all = jax.random.uniform(t_key, (N,), minval=t_min, maxval=t_max)

        x_t_all    = np.zeros((N, n, d), dtype=np.float32)
        s_true_all = np.zeros((N, n, d), dtype=np.float32)

        n_batches = (N + B - 1) // B
        t_start = time.time()
        for b in range(n_batches):
            start, end = b * B, min((b+1)*B, N)
            x0_batch = jnp.array(train_data[start:end])
            t_batch  = t_all[start:end]
            rng, nk  = jax.random.split(rng)
            x_t_b, s_true_b = prepare_batch(manifold, sde, x0_batch, t_batch, nk)
            x_t_all[start:end]    = np.array(x_t_b)
            s_true_all[start:end] = np.array(s_true_b)
            if (b+1) % 50 == 0 or b == n_batches-1:
                elapsed = time.time() - t_start
                frac    = (b+1) / n_batches
                eta     = elapsed / frac * (1 - frac)
                print(f"    batch {b+1}/{n_batches}  ({frac*100:.0f}%)  ETA {eta/60:.1f} min",
                      end="\r", flush=True)

        np.savez_compressed(out_path, x_t=x_t_all, s_true=s_true_all, t=np.array(t_all))
        size_mb = os.path.getsize(out_path) / 1e6
        print(f"\n    Saved {out_path}  ({size_mb:.0f} MB)")
        return rng

    N_REPEATS = 10
    T_MIN, T_MAX = 0.01, 0.99
    rng = jax.random.PRNGKey(2024)
    for r in range(N_REPEATS):
        out_path = os.path.join(PRECOMPUTED_DIR, f"noised_r{r:02d}.npz")
        if os.path.exists(out_path):
            print(f"Repeat {r:2d}: already exists, skipping.")
            continue
        print(f"Repeat {r:2d}/{N_REPEATS-1}...")
        t0 = time.time()
        rng = precompute_one_repeat(manifold, sde, train_data, T_MIN, T_MAX, rng, out_path)
        print(f"  Done in {(time.time()-t0)/60:.1f} min")

    precomp_files = sorted(glob.glob(os.path.join(PRECOMPUTED_DIR, "noised_r*.npz")))
    print(f"\nAll {len(precomp_files)} repeats done.")

REPROCESS_DATA=False — skipping precomputation.
Found 10 existing file(s) in /content/drive/MyDrive/thesis-data/precomputed/bba_v2
  x_t:    (63000, 28, 3)  (float32)
  s_true: (63000, 28, 3)
  t:      range [0.01, 0.99]


## §5. Verify new precomputed data

Quick checks: correct shapes, no NaN, vertical leakage < 1e-6.

In [24]:
import glob

precomp_files = sorted(glob.glob(os.path.join(PRECOMPUTED_DIR, "noised_r*.npz")))
print(f"Found {len(precomp_files)} repeat files.")

s = np.load(precomp_files[0])
print(f"  x_t    : {s['x_t'].shape}  {s['x_t'].dtype}")
print(f"  s_true : {s['s_true'].shape}")
print(f"  t      : range [{s['t'].min():.3f}, {s['t'].max():.3f}]")
assert not np.isnan(s['x_t']).any()    and not np.isnan(s['s_true']).any(), "NaN!"
print("  No NaN. ✓")

# Vertical leakage check on 64 samples from repeat 0
N_check = 64
x_t_check  = jnp.array(s['x_t'][:N_check])    # (N_check, n, d)
s_true_check = jnp.array(s['s_true'][:N_check])

def vertical_leakage(x_t, s_true):
    """Fraction of s_true norm that is vertical (rigid body)."""
    s_horiz = manifold.horizontal_projection_tvector(
        x_t[:, None, :, :],
        s_true[:, None, None, :, :]
    )[:, 0, 0]  # (N, n, d)
    vert = s_true - s_horiz
    leakage = jnp.linalg.norm(vert.reshape(len(x_t), -1), axis=-1) / \
              (jnp.linalg.norm(s_true.reshape(len(x_t), -1), axis=-1) + 1e-12)
    return leakage

leakage = np.array(vertical_leakage(x_t_check, s_true_check))
print(f"\nVertical leakage of s_true (N={N_check}):")
print(f"  mean = {leakage.mean():.2e}   max = {leakage.max():.2e}")
if leakage.max() < 1e-5:
    print("  ✓ Leakage at machine precision — score_target fix confirmed.")
else:
    print("  !! Unexpected leakage — check score_target in manifold_sde.py !!")

Found 10 repeat files.
  x_t    : (63000, 28, 3)  float32
  s_true : (63000, 28, 3)
  t      : range [0.010, 0.990]


AssertionError: NaN!

## §6. Benchmark Phase B step

In [ ]:
import optax, time

BATCH_SIZE  = 64
LR          = 3e-4
HIDDEN_DIMS = (256, 256, 256, 256)

model      = TangentScoreModel(hidden_dims=HIDDEN_DIMS)
optimizer  = optax.chain(optax.clip_by_global_norm(1.0), optax.adam(LR))
train_step = make_train_step(model, manifold, sde, optimizer)

rng = jax.random.PRNGKey(42)
rng, init_key = jax.random.split(rng)
params    = model.init(init_key, jnp.zeros((1, n*d)), jnp.zeros((1, 1)))
ema_p     = params
opt_state = optimizer.init(params)

x_t_b    = jnp.array(s["x_t"][:BATCH_SIZE])
s_true_b = jnp.array(s["s_true"][:BATCH_SIZE])
t_b      = jnp.array(s["t"][:BATCH_SIZE])

print("Warming up JIT...")
params, ema_p, opt_state, loss0 = train_step(params, ema_p, opt_state, x_t_b, s_true_b, t_b)
jax.block_until_ready(loss0)
print(f"  warm-up loss: {float(loss0):.4f}")

times = []
for _ in range(20):
    t0_b = time.time()
    params, ema_p, opt_state, loss0 = train_step(params, ema_p, opt_state, x_t_b, s_true_b, t_b)
    jax.block_until_ready(loss0)
    times.append(time.time() - t0_b)

ms_per_step = np.mean(times) * 1000
steps_per_epoch = len(train_data) // BATCH_SIZE
secs_per_epoch  = np.mean(times) * steps_per_epoch
print(f"\nPhase B step (B={BATCH_SIZE}): {ms_per_step:.2f} ms")
print(f"Steps/epoch: {steps_per_epoch}  →  {secs_per_epoch:.1f} s/epoch")
print(f"Est. 3000 epochs: {3000*secs_per_epoch/60:.1f} min")

## §7. Train

Uses `likelihood_weighting=False` (uniform weight per time step) to ensure the model
receives equal gradient signal across the full diffusion trajectory, not just at small `t`.

In [ ]:
import json

N_EPOCHS    = 3000
BATCH_SIZE  = 64
LR          = 3e-4
HIDDEN_DIMS = (256, 256, 256, 256)
LOG_EVERY   = 50
LIKELIHOOD_WEIGHTING = False  # uniform weight — see Phase 3.6 postmortem

model = TangentScoreModel(hidden_dims=HIDDEN_DIMS)

print(f"Training: {N_EPOCHS} epochs, B={BATCH_SIZE}, lr={LR}")
print(f"likelihood_weighting={LIKELIHOOD_WEIGHTING}  (uniform across t)")
print(f"Checkpoints → {LOCAL_CKPT}  (synced to Drive in §10)\n")

state, history = train_from_precomputed(
    model=model,
    manifold=manifold,
    sde=sde,
    precomputed_dir=PRECOMPUTED_DIR,
    n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LR,
    grad_clip=1.0,
    ema_decay=0.995,
    likelihood_weighting=LIKELIHOOD_WEIGHTING,
    seed=42,
    log_every=LOG_EVERY,
    ckpt_dir=LOCAL_CKPT,
)

with open(os.path.join(LOCAL_CKPT, "loss_history.json"), "w") as f:
    json.dump(history, f)

first_loss = history[0][1]
last_loss  = history[-1][1]
print(f"\nInitial: {first_loss:.4f}  →  Final: {last_loss:.4f}  ({last_loss/first_loss*100:.1f}% of initial)")

## §8. Loss curve

In [ ]:
import matplotlib.pyplot as plt

epochs = [ep   for ep, _    in history]
losses = [loss for _,  loss in history]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs, losses, linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("DSM Loss (uniform weight)")
ax.set_title(f"BBA Phase 3 v2 — TangentScoreModel{HIDDEN_DIMS}, n={n} Cα, α=1.0")
ax.axhline(last_loss, color="r", linestyle="--", alpha=0.5,
           label=f"Final {last_loss:.2f} ({last_loss/first_loss*100:.0f}% of initial)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(LOCAL_CKPT, "loss_curve.png"), dpi=150)
plt.show()
print("Loss curve saved.")

## §9. Score sanity check — cosine similarity by t-regime

Expect `cos_sim > 0.5` in the mid-t regime after convergence.  
The Phase 3.6 run had `cos_sim ≈ 0` across all regimes (H-bug + wrong precomputed data).

In [ ]:
params   = state["ema_params"]
score_fn = lambda x_flat, t_col: model.apply(params, x_flat, t_col)

def project_h_single(x_frame, v):
    """Project tangent vector v at frame x to horizontal space."""
    return manifold.horizontal_projection_tvector(
        x_frame[None, None],   # (1, 1, n, d)
        v[None, None, None],   # (1, 1, 1, n, d)
    )[0, 0, 0]                 # (n, d)

def cosine_check(t_min, t_max, label, B=128):
    rng_ = jax.random.PRNGKey(42)
    rng_, tk_, nk_ = jax.random.split(rng_, 3)
    x0_ = jnp.array(test_data[:B])
    t_  = jax.random.uniform(tk_, (B,), minval=t_min, maxval=t_max)
    x_t_, s_true_ = prepare_batch(manifold, sde, x0_, t_, nk_)

    s_pred_ = score_fn(x_t_.reshape(B, n*d), t_.reshape(B, 1)).reshape(B, n, d)
    s_pred_h_ = jax.vmap(project_h_single)(x_t_[:, None], s_pred_)

    a = s_pred_h_.reshape(B, -1)
    b = s_true_.reshape(B, -1)
    cos = (a * b).sum(-1) / (jnp.linalg.norm(a, axis=-1) * jnp.linalg.norm(b, axis=-1) + 1e-8)
    nr  = jnp.linalg.norm(a, axis=-1) / (jnp.linalg.norm(b, axis=-1) + 1e-8)
    return float(cos.mean()), float(cos.std()), float(nr.mean())

print("Score cosine similarity vs t-regime (s_pred projected to horizontal space):")
print(f"{'t range':<20} {'cos_sim mean':>14} {'cos_sim std':>12} {'||pred||/||true||':>18}")
print("-" * 66)
for t_min, t_max, label in [
    (0.01, 0.10, "small t  (t<0.1)"),
    (0.10, 0.30, "mid-low  (0.1-0.3)"),
    (0.30, 0.70, "mid      (0.3-0.7)"),
    (0.70, 0.99, "large t  (t>0.7)"),
]:
    mu, sd, nr = cosine_check(t_min, t_max, label)
    flag = "✓" if mu > 0.3 else ("~" if mu > 0.1 else "✗")
    print(f"{label:<20}  {mu:>+.4f} ± {sd:.4f}   {nr:>10.4f}   {flag}")

print("\nInterpretation:")
print("  cos_sim > 0.5 → model learned useful score direction")
print("  cos_sim ≈ 0   → model output is not aligned with true score")
print("  cos_sim < 0   → something is wrong (check projection convention)")

## §10. Save checkpoint and results to Drive

In [ ]:
import shutil

os.makedirs(DRIVE_CKPT, exist_ok=True)

for fname in ["ckpt_final.npz", "loss_history.json", "loss_curve.png"]:
    src = os.path.join(LOCAL_CKPT, fname)
    if os.path.exists(src):
        dst = os.path.join(DRIVE_CKPT, fname)
        shutil.copy(src, dst)
        size_kb = os.path.getsize(dst) / 1024
        print(f"  {fname} → Drive  ({size_kb:.0f} KB)")
    else:
        print(f"  {fname}: NOT FOUND (training may not have run yet)")

print(f"\nAll results at: {DRIVE_CKPT}")

## §11. Download results to local machine

After the notebook finishes, pull the checkpoint and precomputed data from Drive to your local machine.

### Option A — rclone (recommended for large precomputed data)

```bash
# Download the new checkpoint:
rclone copy \
  gdrive:thesis-data/runs/bba_phase3_v2/ \
  /Users/I745505/private/thesis/pocs/manifold-scoremd/riemannian-scoremd/runs/bba_phase3_v2/ \
  --progress

# Download the new precomputed data (optional — large, ~1.7 GB for 10 repeats):
rclone copy \
  gdrive:thesis-data/precomputed/bba_v2/ \
  /Users/I745505/private/thesis/pocs/manifold-scoremd/riemannian-scoremd/data/precomputed/bba_v2/ \
  --progress
```

### Option B — Google Drive web UI

1. Open [drive.google.com](https://drive.google.com)
2. Navigate to `My Drive / thesis-data / runs / bba_phase3_v2`
3. Select all files → right-click → Download (creates a zip)
4. Unzip into `riemannian-scoremd/runs/bba_phase3_v2/`

### Option C — gdown (in a local terminal)

```bash
pip install gdown
# Get the folder ID from the Drive URL (the string after /folders/)
gdown --folder https://drive.google.com/drive/folders/FOLDER_ID -O runs/bba_phase3_v2/
```

### Verifying the download

```bash
# From the project root:
source diepeveen2024/.venv/bin/activate
python -c "
import numpy as np
ckpt = np.load('riemannian-scoremd/runs/bba_phase3_v2/ckpt_final.npz', allow_pickle=True)
print('Keys:', list(ckpt.keys()))
import json
h = json.load(open('riemannian-scoremd/runs/bba_phase3_v2/loss_history.json'))
print(f'Loss: {h[0][1]:.2f} → {h[-1][1]:.2f}')
"
```